# CAREPILOT — Similar Ticket Retrieval Benchmark

*Chapter 2 — État de l'art et benchmarking IA · CAREPILOT operational AI*

This notebook documents the **Task 7** retrieval benchmark: surfacing historically resolved tickets that match a new operational incident using NLP over the full historical corpus.

## 1. Objective

Help N1/N2 agents resolve incidents faster by retrieving **similar resolved tickets** from the historical corpus. The system ranks past tickets by textual and operational similarity, then surfaces resolution patterns, root-cause families, and peer cases for decision support.

This benchmark compares three retrieval approaches in order to select the most suitable method for integration into CAREPILOT.

## 2. Problem Type

- **Semantic / lexical retrieval** (not classification)
- **Query:** new ticket (title, description, perimeter, category, resolution metadata)
- **Corpus:** full labeled historical ticket set (`ml_training.historical_tickets`)
- **Output:** ranked list of similar historical tickets (top-K)
- **Ground truth:** tickets sharing the same `similar_case_group_id` (excluding the query ticket itself)

## 3. Dataset Description

| Property | Value |
| --- | --- |
| Source | `ml_training.historical_tickets` |
| Corpus size | 14,000 tickets |
| Tickets with group id | 14,000 |
| Unique similar-case groups | 792 |
| Group size (min / avg / max) | 9 / 17.7 / 35 |
| Evaluable test queries | 1,584 |

**Group metadata:** `incident_family`, `root_cause_family`, `resolution_pattern`

**Retrieval text per ticket:** title + description + perimeter + incident category + resolution summary + incident/root-cause/resolution families + derived keywords, with operational synonym normalization (POS, SCO, VLAN, EMV, etc.).

## 4. Preprocessing

1. **Text normalization** — lowercase, punctuation cleanup, stopword removal (lexical methods)
2. **Synonym expansion** — operational shorthand (POS, SCO, payment authorization, printer, network)
3. **Field assembly** — ticket text combines structured metadata and free-text fields
4. **Corpus indexing** — TF-IDF matrix, BM25 inverted index, or sentence-transformer embeddings depending on method
5. **Query exclusion** — the query ticket itself is removed from candidate results

## 5. Methods Compared

1. **TF-IDF + Cosine Similarity** — sparse lexical baseline; strong when query tokens overlap historical tickets directly
2. **BM25** — probabilistic lexical retrieval; better ranking for short operational queries and keyword-heavy tickets
3. **Sentence Transformers (`all-MiniLM-L6-v2`)** — dense semantic embeddings over combined ticket text; evaluated with chunk-level pooling on the historical corpus

Unlike KM retrieval (Task 6), similar-ticket groups are **large and heterogeneous** — lexical overlap often outperforms semantic embeddings when group labels are broad.

## 6. Evaluation Metrics

- **Precision@1 (P@1)** — is the top retrieved ticket relevant?
- **Precision@3 (P@3)** — relevant hits in top 3 divided by min(3, |relevant|)
- **Recall@3 (R@3)** — fraction of all relevant peers found in top 3
- **MRR** — mean reciprocal rank of the first relevant hit
- **nDCG@3** — ranking quality with position discount
- **Avg retrieval time (ms)** — operational latency per query

## 7. Retrieval Pipeline

```
historical_tickets corpus
        → preprocess + index (TF-IDF / BM25 / embeddings)
        → test query ticket
        → rank corpus (exclude self)
        → top-K similar tickets
        → compare to similar_case_group_id ground truth
```

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "ml" / "artifacts").exists():
        ROOT = parent
        break
else:
    raise FileNotFoundError("Project root not found")

SHARED = ROOT / "ml" / "notebooks" / "_shared"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from benchmark_notebook_helpers import display_plot, display_plots, find_project_root, plot_results_table

ROOT = find_project_root()
ARTIFACTS = ROOT / "ml" / "artifacts"
PLOTS = ROOT / "ml" / "plots"
REPORTS = ROOT / "ml" / "reports"
BENCHMARKS = ROOT / "ml" / "benchmarks"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

RUN_BENCHMARK = False



In [ ]:
script = BENCHMARKS / "similar_ticket_retrieval" / "benchmark.py"
if RUN_BENCHMARK:
    subprocess.run([sys.executable, str(script), "--project-root", str(ROOT)], check=True)
else:
    print("Loading saved benchmark artifacts (set RUN_BENCHMARK=True to re-run).")
    print(f"  python {script} --project-root {ROOT}")



## 8. Benchmark Execution — Load Results

In [ ]:
metrics_path = ARTIFACTS / "metrics" / "similar_ticket_retrieval_metrics.json"
payload = json.loads(metrics_path.read_text(encoding="utf-8"))
dataset = payload["dataset"]
validation = dataset["validation"]

dataset_summary = pd.DataFrame(
    {
        "Metric": [
            "Corpus tickets",
            "Test queries",
            "Unique similar-case groups",
            "Avg group size",
            "Min / max group size",
        ],
        "Value": [
            dataset["corpus_tickets"],
            dataset["test_queries"],
            validation["unique_groups"],
            round(validation["avg_group_size"], 1),
            f"{validation['min_group_size']} / {validation['max_group_size']}",
        ],
    }
)
display(Markdown("### Dataset validation"))
display(dataset_summary)

rows = payload["results"]
results_df = pd.DataFrame(rows)
results_df = results_df.rename(
    columns={
        "method": "Method",
        "precision_at_1": "P@1",
        "precision_at_3": "P@3",
        "recall_at_3": "R@3",
        "mrr": "MRR",
        "ndcg_at_3": "nDCG@3",
        "avg_retrieval_time_ms": "Avg Time (ms)",
    }
)
for col in ["P@1", "P@3", "R@3", "MRR", "nDCG@3"]:
    results_df[col] = results_df[col].map(lambda x: f"{x:.4f}")
results_df["Avg Time (ms)"] = results_df["Avg Time (ms)"].map(lambda x: f"{x:.1f}")

display(Markdown("### Retrieval metrics (held-out test queries)"))
display(results_df)



## 9. Benchmark Results — Report Table

In [ ]:
table_df = pd.DataFrame(rows)
figure_df = table_df.rename(
    columns={
        "method": "Method",
        "precision_at_1": "P@1",
        "precision_at_3": "P@3",
        "recall_at_3": "R@3",
        "mrr": "MRR",
        "ndcg_at_3": "nDCG@3",
        "avg_retrieval_time_ms": "Avg Time (ms)",
    }
)
for col in ["P@1", "P@3", "R@3", "MRR", "nDCG@3"]:
    figure_df[col] = figure_df[col].map(lambda x: f"{x:.4f}")
figure_df["Avg Time (ms)"] = figure_df["Avg Time (ms)"].map(lambda x: f"{x:.0f} ms")

png_path = PLOTS / "03_similar_ticket_benchmark_results_table.png"
plot_results_table(
    figure_df,
    title="CarePilot Similar Ticket Retrieval Benchmark",
    footer="1,584 test queries · Corpus: 14,000 tickets · Ground truth: similar_case_group_id",
    output_path=png_path,
)
display_plot(png_path)



## 10. Visualizations

In [ ]:
display_plots(
    [
        PLOTS / "03_similar_ticket_method_comparison.png",
        PLOTS / "03_similar_ticket_precision_at_k.png",
        PLOTS / "03_similar_ticket_mrr_ndcg.png",
        PLOTS / "03_similar_ticket_latency.png",
        PLOTS / "03_similar_ticket_cluster_distribution.png",
        PLOTS / "03_similar_ticket_success_cases.png",
        PLOTS / "03_similar_ticket_failure_cases.png",
    ]
)



## 11. Interpretation

**BM25 achieves the best overall retrieval quality** on this benchmark (P@1 ≈ 0.57, MRR ≈ 0.65, nDCG@3 ≈ 0.49). Operational tickets in the same `similar_case_group_id` often share **exact keywords** (error codes, lane numbers, perimeter names) that BM25 and TF-IDF capture reliably.

- **TF-IDF + Cosine** (P@1 ≈ 0.36) — usable baseline but weaker ranking when queries are short or vocabulary diverges from historical wording.
- **BM25** — best precision and MRR; higher latency (~591 ms) due to corpus scoring cost on 14k tickets.
- **Sentence Transformers** underperform here (P@1 ≈ 0.27) because similar-case groups are **broad operational clusters** — semantic paraphrase helps KM article matching (Task 6) more than large heterogeneous ticket groups.

**Operational meaning:** agents receive peer tickets that genuinely share incident family and resolution pattern, supporting faster root-cause identification without replacing deterministic AI predictions from Tasks 1–6.

## 12. Selected Method

**Selected for CAREPILOT benchmark & reference implementation: BM25**

- Artifact: `ml/artifacts/models/similar_ticket_bm25.joblib`
- Supporting artifacts: `similar_ticket_tfidf.joblib`, `similar_ticket_retrieval_minilm.joblib`
- Metrics: `ml/artifacts/metrics/similar_ticket_retrieval_metrics.json`

*Note:* Production demo UI may use seeded preview data for stability; this benchmark reflects offline retrieval quality on the full historical corpus.

## 13. Limitations

- Similar-case groups are large (avg ~18 tickets) — Recall@3 stays low because many relevant peers exist beyond top-3
- Wording inconsistency across stores reduces semantic embedding performance
- Corpus must stay synchronized with resolved-ticket history
- Retrieval latency scales with corpus size; indexing strategy matters at production volume
- Benchmark evaluates offline group membership, not agent satisfaction or resolution time saved

## 14. Generated Artifacts

- `ml/artifacts/metrics/similar_ticket_retrieval_metrics.json`
- `ml/artifacts/models/similar_ticket_bm25.joblib`
- `ml/plots/03_similar_ticket_benchmark_results_table.png`
- `ml/plots/03_similar_ticket_method_comparison.png`
- `ml/plots/03_similar_ticket_precision_at_k.png`
- `ml/plots/03_similar_ticket_mrr_ndcg.png`
- `ml/plots/03_similar_ticket_latency.png`
- `ml/reports/03_similar_ticket_retrieval_summary.md`